
# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: If any overlay has `stylized=True`, the hexes layer is rendered with organic region boundaries via `grid.styleLayer()`. Otherwise `grid.update()` is called, flushing hex style mutations into the `hexes` layer as flat polygons.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

**Stylized note:** Only the *first* overlay (by priority) with `stylized=True` controls the edge function. Multiple stylized overlays don't compose — one wins.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay(stylized, edge_fn)` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay(stylized, edge_fn)` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `WatershedOverlay()` | 35 | SVG | Watershed boundary dots (requires `result`) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width, lake_base_size)` | 60 | SVG | River network from drainage basins |
| `SquadVisionOverlay(squads, num_turns, ...)` | 65 | SVG | Vision cones + facing bars (game-specific) |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Stylized mode — organic region borders

Pass `stylized=True` to any hex-style overlay to replace flat hex polygons with Gosper-curve organic boundaries:

```python
# Organic edges on terrain colors
TerrainDisplay(TerrainOverlay(stylized=True), terrain=t)

# Organic edges on parchment
TerrainDisplay(CreamOverlay(stylized=True), terrain=t)

# Custom edge function
TerrainDisplay(
    CreamOverlay(stylized=True, edge_fn=unique_windy_edge(iterations=4)),
    terrain=t
)
```

Without `stylized=True`, hexes render as clean polygons. With it, region boundaries get a hand-drawn feel via `grid.styleLayer()`.

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` (coarse mode) |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | `WatershedOverlay`, overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |
| `region=` | triggers `terrain.zoom(region, padding)` | Zoomed-in views |
| `padding=` | passed to `zoom()` (default `1`) | Border hexes around region |
| `radius=` | calls `grid.adjustRadius()` | Custom hex sizing |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'} — but actually requires=set(), checks internally
# If neither basins nor coarse_basins+c2f are available, returns ""
```

---

## Zoom & Projection

When you pass `region=` to `TerrainDisplay`, the system creates a zoomed-in view:

```python
TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(max_width=6),
    terrain=myTerr,
    basins=basins,
    region=my_region,   # ← triggers zoom
    padding=2
)
```

### What happens internally

1. `terrain.zoom(region, padding)` creates a new fine-resolution `Terrain` cropped to the region, with a `c2f` mapping (coarse index → list of fine indices).
2. The original `basins` reference the coarse grid, so they're stashed as `ctx.coarse_basins`. The `ctx.basins` field is set to `None` to prevent overlays from accidentally using stale coarse indices.
3. `c2f` is stored in `ctx.c2f` for overlays that need to project coarse data onto the fine grid.

### How RiverOverlay handles zoom

`RiverOverlay` detects the zoomed context (`c2f is not None and coarse_basins is not None`) and switches to projection mode:

- Iterates coarse river segments from `coarse_basins`
- Maps each coarse hex index to its fine grid position via `c2f`
- Draws the river path using fine grid coordinates with width scaling preserved from the coarse flow data
- Projects lake basins the same way

This avoids recomputing drainage on the fine grid (which would be O(n²)) and keeps rivers visually consistent between zoom levels.

---

## Creating Custom Overlays

### OverlaySpec fields

```python
@dataclass
class OverlaySpec:
    name: str                     # Unique layer name
    renderer: Callable            # fn(ctx: OverlayContext) -> str
    requires: set = set()         # Context keys needed (skip if missing)
    priority: int = 50            # Z-order (lower = bottom)
    stylized: bool = False        # Use organic region boundaries
    edge_fn: Callable = None      # Edge transform for stylized mode
```

### SVG overlay example

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Hex style mutation example

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## Testing with OverlayContext

Use `OverlayContext.from_terrain()` and `.test()` to debug overlays outside of `TerrainDisplay`:

```python
# Build a standalone context
ctx = OverlayContext.from_terrain(myTerr, basins=basins, soil=soil)

# Run overlays — errors propagate (not swallowed), prints status per overlay
ctx.test(CreamOverlay(), RiverOverlay())
#   ✓ cream                pri=10   style-only
#   ✓ rivers               pri=60   SVG (4823 chars)
```

`from_terrain()` accepts the same keywords as `TerrainDisplay` (`region=`, `padding=`, `radius=`, `basins=`, `soil=`, `result=`, `board=`, `c2f=`), so you can test zoom scenarios too:

```python
ctx = OverlayContext.from_terrain(myTerr, basins=basins, region=my_region, padding=2)
ctx.test(CreamOverlay(), RiverOverlay())
```

Unlike `TerrainDisplay` which logs warnings and skips, `.test()` raises `ValueError` on missing dependencies — fail fast for debugging.

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Stylized edges | `CreamOverlay(stylized=True)` or `TerrainOverlay(stylized=True)` |
| Custom edge function | `CreamOverlay(stylized=True, edge_fn=unique_windy_edge(iterations=4))` |
| Zoom into a region | `region=my_region, padding=2` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Test standalone | `OverlayContext.from_terrain(t).test(SomeOverlay())` |
| Create custom | Write a function returning `OverlaySpec` |
```

In [ ]:
#| default_exp overlay

In [ ]:
#| export
#| hide
#import nbdev; nbdev.nbdev_export()
import sys
import math
from fastcore.basics import patch

#| export
## Getting Started

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex


from HexMagic.styles import StyleCSS,  SVGBuilder, SVGLayer

In [ ]:
#| export
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain


Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, SoilSystem


In [ ]:
#| export
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.climate import TerrainFactory

In [ ]:
#| export

from HexMagic.terraform import Terraform
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.core import TerraDemo
from HexMagic.primitives import HexTouchMap, HexRegion
from HexMagic.water.soil import SoilSystem, SoilType


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass

In [ ]:
#!cat ../docs/ll*

In [ ]:
#read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

Can you start to build TerrainDisplay

In [ ]:
#| export
#| export
import logging
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class OverlaySpec:
    """Specification for a single overlay layer."""
    name: str
    renderer: Callable[['OverlayContext'], str]
    requires: set = field(default_factory=set)
    priority: int = 50
    stylized: bool = False       # Use organic region boundaries instead of flat hexes
    edge_fn: Callable = None     # Edge transform for stylized mode (default: unique_windy_edge)


In [ ]:
#| export
@dataclass
class OverlayContext:
    """Rendering context passed to overlay renderers."""
    terrain: 'Terrain'
    grid: 'HexGrid'
    builder: 'SVGBuilder'
    extras: dict = field(default_factory=dict)

    def __getattr__(self, name):
        if name in self.__dict__.get('extras', {}):
            return self.extras[name]
        raise AttributeError(f"OverlayContext has no '{name}' — did you forget to pass it?")

    @property
    def available(self):
        keys = {'terrain', 'grid', 'builder'}
        keys |= {k for k, v in self.extras.items() if v is not None}
        return keys

    @classmethod
    def from_terrain(cls, terrain, *, region=None, padding=1, radius=None,
                    basins=None, soil=None, result=None, board=None, c2f=None):
        """Build a test context — same logic as TerrainDisplay but standalone."""
        orig_terrain = terrain

        if region is not None:
            terrain = terrain.zoom(region, padding=padding)
            if c2f is None:
                c2f = getattr(terrain, 'c2f', None)

            # If no board provided, create a minimal one so overlays
            # can simulate on the original (coarse) grid
            if board is None:
                board = type('_Board', (), {'terrain': orig_terrain})()

        grid = terrain.hexGrid
        if radius:
            grid.adjustRadius(radius)

        if basins is None and result and hasattr(result, 'basins'):
            basins = result.basins
        if soil is None and result and hasattr(result, 'soil'):
            soil = result.soil

        return cls(
            terrain=terrain, grid=grid, builder=grid.builder,
            extras=dict(result=result, board=board, c2f=c2f,
                        basins=basins, soil=soil)
        )


    def test(self, *overlays: 'OverlaySpec'):
        """Run overlays with errors propagated — no swallowing."""
        results = {}
        for o in sorted(overlays, key=lambda o: o.priority):
            missing = o.requires - self.available
            if missing:
                raise ValueError(f"'{o.name}' requires {missing} but context has {self.available}")
            svg = o.renderer(self)
            tag = f"SVG ({len(svg)} chars)" if svg else "style-only"
            print(f"  ✓ {o.name:20s} pri={o.priority:<4d} {tag}")
            results[o.name] = svg
        return results


In [ ]:
#| export
@patch
def zoom(self: Terrain, region: HexRegion, padding: int = 1) -> 'Terrain':
    """Create a new Terrain zoomed into a HexRegion, with c2f mapping."""
    new_grid, _, mapper = region.crop_to_centered_grid(
        style=self.hexGrid.style, padding=padding
    )
    if new_grid is None:
        zoomed = self.clone()
        zoomed.c2f = None
        return zoomed

    zoomed = self.clone()
    zoomed.assign_grid(new_grid)

    # Build reverse mapping: old_idx → [new_idx] (list to match _map_point API)
    c2f = {}
    for new_idx in range(len(new_grid.hexes)):
        old_idx = mapper(new_idx)
        if 0 <= old_idx < len(self.elevations):
            zoomed.elevations[new_idx] = self.elevations[old_idx]
            for k in self.fields:
                zoomed.fields[k][new_idx] = self.fields[k][old_idx]
            c2f[old_idx] = [new_idx]

    zoomed.c2f = c2f
    return zoomed


In [ ]:
#| export
def TerrainDisplay(
    *overlays: OverlaySpec,
    terrain: 'Terrain',
    result=None, board=None, c2f=None,
    basins=None, soil=None,
    radius=None, debug=False,
    region: 'HexRegion' = None,
    padding: int = 1,
    **attrs
):
    coarse_basins = None

    # Zoom if region provided
    if region is not None:
        # Basins reference coarse grid — stash them separately
        if basins is not None:
            coarse_basins = basins
            basins = None  # prevent overlays using stale coarse indices

        terrain = terrain.zoom(region, padding=padding)
        if c2f is None:
            c2f = getattr(terrain, 'c2f', None)

    grid = terrain.hexGrid
    if radius: grid.adjustRadius(radius)

    if basins is None and result and hasattr(result, 'basins'):
        basins = result.basins
    if soil is None and result and hasattr(result, 'soil'):
        soil = result.soil

    ctx = OverlayContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        extras=dict(result=result, board=board, c2f=c2f,
                    basins=basins, soil=soil, coarse_basins=coarse_basins)
    )

    available = {'terrain', 'grid', 'builder'}
    if soil:           available.add('soil')
    if result:         available.add('result')
    if board:          available.add('board')
    if c2f:            available.add('c2f')
    if basins:         available.add('basins')
    if coarse_basins:  available.add('coarse_basins')

    # Phase 1: Run renderers, collect SVG strings (+ hex style side effects)
    collected = []
    sorted_overlays = sorted(overlays, key=lambda o: o.priority)
    for spec in sorted_overlays:
        missing = spec.requires - available
        if missing:
            logging.warning(f"Skipping {spec.name}: missing {missing}")
            continue
        try:
            svg = spec.renderer(ctx)
            if svg: collected.append((spec.name, svg))
        except Exception as e:
            logging.error(f"Overlay {spec.name} failed: {e}")

    # Phase 2: Bake hex styles
    stylized_spec = next((s for s in sorted_overlays if s.stylized), None)
    if stylized_spec:
        f = stylized_spec.edge_fn or unique_windy_edge()
        hexes_svg = grid.styleLayer(f=f, smooth=True)
        grid.builder.adjust("hexes", hexes_svg)
    else:
        grid.update()

    # Phase 3: Add SVG overlay layers ON TOP of hexes
    for name, svg in collected:
        grid.builder.adjust(name, svg)

    if debug: return grid.builder.__ft__()

    return Div(
        HexTouchMap(grid, cls=attrs.pop('map_cls', 'w-full h-full')),
        **attrs
    )


In [ ]:
myTerr = TerraDemo().japan_korea_map()

In [ ]:
myTerr.compute_climate()

In [ ]:
myTerr.carve_to_ocean(num_lakes=0)
basins = DrainageBasins(myTerr)

In [ ]:
#| export
#| export
def TerrainOverlay(stylized=False, edge_fn=None, **kw) -> OverlaySpec:
    """Color hexes by elevation. Pass stylized=True for organic region borders."""
    def render(ctx: OverlayContext) -> str:
        ctx.terrain.colorMap()
        return ""
    return OverlaySpec("heights", render, priority=5, stylized=stylized, edge_fn=edge_fn)


In [ ]:
#| export
#| export
def ClimateOverlay(levels: int = 5, min_density: float = 0.25, **kw) -> OverlaySpec:
    """Climate zone dots colored by climate type, density by precipitation."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.dottedClimate(flow_levels=levels, min_density=min_density)
    return OverlaySpec("climate", render, priority=40)



def WatershedOverlay(**kw) -> OverlaySpec:
    """Watershed boundary dot overlay."""
    def render(ctx: OverlayContext) -> str:
        if not ctx.result or not hasattr(ctx.result, 'basins') or not ctx.result.basins:
            return ""
        return ctx.result.basins.dotted_watershed_overlay()
    return OverlaySpec("watersheds", render, requires={'result'}, priority=35)


In [ ]:
!cat ../HexMagic/database.py

The error is that `CountryFlag` doesn't have `register_glyph_styles` or `glyphs` methods. Those live on `DiagramGlyphs`, which is a separate dataclass wrapping a flag. Here's the fixed overlay:

```python
def SquadVisionOverlay(squads, num_turns=5, facing_only=True,
                       opacity=0.30, color_attr="darkPrimary",
                       elevation_mult=0.005, **kw) -> OverlaySpec:
    """Vision cones + facing bars at each piece's simulated final position."""

    def render(ctx) -> str:
        grid    = ctx.grid
        builder = ctx.builder
        c2f     = getattr(ctx, 'c2f', None)

        board = getattr(ctx, 'board', None)
        if board is not None:
            sim_grid = board.terrain.hexGrid
            sim_elev = board.terrain.elevations
            sim_ctry = board.terrain.fields.get("country")
        else:
            sim_grid = grid
            sim_elev = ctx.terrain.elevations
            sim_ctry = ctx.terrain.fields.get("country")

        # ── helper: coarse index → list of fine indices ──
        def _c2f_list(idx):
            if c2f is None:
                return [idx]
            val = c2f.get(idx)
            if val is None:
                return []
            return val if isinstance(val, list) else [val]

        def _c2f_first(idx):
            mapped = _c2f_list(idx)
            return mapped[0] if mapped else -1

        # ── Simulate to collect final positions ──
        flag_hexes: dict[str, tuple[object, set[int]]] = {}
        bars: list[tuple[int, int, object]] = []

        r       = grid.radius
        dot_r   = max(2.5, r * 0.18)
        spacing = max(5.0, r * 0.32)
        N       = len(grid.hexes)

        for squad in squads:
            pieces = squad.alive if hasattr(squad, 'alive') else squad.pieces
            for piece in pieces:
                if not piece.instructions or not piece.instructions.rules:
                    continue
                if not piece.flag:
                    continue

                steps, _ = piece.plan_region(
                    sim_grid, sim_elev,
                    num_turns=num_turns, padding=0,
                    countries=sim_ctry,
                )
                if not steps:
                    continue

                last = steps[-1]
                key  = piece.flag.name

                elev      = max(0, sim_elev[last.hex_idx])
                eff_sight = int(piece.sight + elev * elevation_mult)

                if facing_only:
                    facing_dir = HexPosition.directions()[last.facing % 6]
                    fov = HexPosition.origin().field_of_view(facing_dir, eff_sight)
                    visible = [idx for hp in fov
                               if (idx := sim_grid.hexposition_to_index(hp, last.hex_idx)) >= 0]
                else:
                    visible = sim_grid.indices_in_range(last.hex_idx, eff_sight)

                if c2f is not None:
                    visible    = [fi for v in visible for fi in _c2f_list(v) if fi >= 0]
                    render_idx = _c2f_first(last.hex_idx)
                else:
                    render_idx = last.hex_idx

                if key not in flag_hexes:
                    flag_hexes[key] = (piece.flag, set())
                flag_hexes[key][1].update(visible)

                if 0 <= render_idx < N:
                    bars.append((render_idx, last.facing, piece.flag))

        # ── Render vision hex fills ──
        parts = []

        for key, (flag, hex_indices) in flag_hexes.items():
            color  = getattr(flag, color_attr, "#888")
            pat_id = f"vision_{key}"

            dot = (f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
                   f'r="{dot_r:.1f}" fill="{color}"/>')
            builder.add_definition(SVGDef(
                "pattern", pat_id, dot,
                x="0", y="0",
                width=f"{spacing:.1f}", height=f"{spacing:.1f}",
                patternUnits="userSpaceOnUse",
            ))

            vis_style = StyleCSS(
                f"vision_{key}",
                fill=f"url(#{pat_id})",
                opacity=str(opacity),
                stroke=color,
                stroke_width="0.6",
                stroke_dasharray="3,2",
                stroke_opacity="0.4",
            )
            builder.add_style(vis_style)

            for idx in sorted(hex_indices):
                if 0 <= idx < N:
                    h = grid.hexes[idx]
                    parts.append(Hex(h.radius, h.center, vis_style, v=h.v).svg())

        # ── Render facing bars via DiagramGlyphs ──
        for render_idx, facing, flag in bars:
            c = grid.hexes[render_idx].center
            glyphs = DiagramGlyphs(flag, size=r * 0.7)
            glyphs.register_styles(builder)
            parts.append(glyphs.facing_bar(c, facing))

        return '\n'.join(parts)

    return OverlaySpec("squad_vision", render, priority=65)
```

The key fix is at the bottom — instead of calling nonexistent methods on `CountryFlag`:

```python
# BROKEN
flag.register_glyph_styles(builder)
glyphs = flag.glyphs(size=r * 0.7)
parts.append(glyphs.facing_bar(c.x, c.y, facing))
```

It now creates a `DiagramGlyphs` instance from the flag and uses its API:

```python
# FIXED
glyphs = DiagramGlyphs(flag, size=r * 0.7)
glyphs.register_styles(builder)
parts.append(glyphs.facing_bar(c, facing))
```

Note that `facing_bar` takes a `MapCord` center (not separate x/y), which `grid.hexes[render_idx].center` already is.

In [ ]:
#| export
def RiverOverlay(top_n=8, simplify_k=3, max_width=None, lake_base_size=3, **kw) -> OverlaySpec:
    """River network — projects coarse paths onto fine grid when zoomed."""

    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        coarse_basins = getattr(ctx, 'coarse_basins', None)

        # ── Zoomed: project coarse river geometry ──
        if c2f is not None and coarse_basins is not None:
            return _render_projected(ctx, coarse_basins, c2f)

        # ── Normal (coarse): draw directly ──
        if not getattr(ctx, 'basins', None):
            return ""
        return ctx.basins.draw_watersheds(
            top_n=top_n, simplify_k=simplify_k,
            max_width=max_width, lake_base_size=lake_base_size
        )

    def _first_fine(c2f, idx):
        """Coarse index → first valid fine index, or -1."""
        mapped = c2f.get(idx)
        if mapped is None: return -1
        return mapped[0] if isinstance(mapped, list) else mapped

    def _render_projected(ctx, coarse_basins, c2f):
        grid = ctx.grid
        N    = len(grid.hexes)
        max_w = max_width or grid.radius * 2 / 3

        watersheds = coarse_basins.get_major(top_n)
        if not watersheds:
            return ""

        # Global max flow for consistent width scaling
        global_max_flow = 0
        for ws in watersheds:
            flows = ws.calculate_flow()
            if flows:
                global_max_flow = max(global_max_flow, max(flows.values()))
        log_max = math.log10(global_max_flow + 1) if global_max_flow > 0 else 1

        parts = []
        for shed in watersheds:
            flows = shed.calculate_flow()
            coarse_terrain = shed.terrain

            # ── Rivers: project each segment ──
            for hexes in shed.segments():
                points = []
                mapped_hexes = []

                for h in hexes:
                    # Ocean terminal — stop, don't try to map
                    if coarse_terrain.elevations[h] <= 0:
                        # Midpoint between last mapped point and ocean
                        # (like segment_to_points does)
                        if points:
                            fine_ocean = _first_fine(c2f, h)
                            if 0 <= fine_ocean < N:
                                oc = grid.hexes[fine_ocean].center
                                lc = points[-1]
                                points.append(MapCord((lc.x + oc.x) / 2,
                                                      (lc.y + oc.y) / 2))
                        break

                    fine_idx = _first_fine(c2f, h)
                    if 0 <= fine_idx < N:
                        points.append(grid.hexes[fine_idx].center)
                        mapped_hexes.append(h)

                if len(points) < 2:
                    continue

                # Width from flow (log scale, same as drawRiver)
                widths = []
                for h in mapped_hexes:
                    flow = max(1, flows.get(h, 1))
                    ratio = math.log10(flow + 1) / log_max if log_max > 0 else 0
                    widths.append(1.0 + ratio * (max_w - 1.0))

                avg_width = sum(widths) / len(widths) if widths else 1.0

                # Windiness from gradient
                if len(mapped_hexes) >= 2:
                    drop = (coarse_terrain.elevations[mapped_hexes[0]]
                          - coarse_terrain.elevations[mapped_hexes[-1]])
                    max_elev = max(coarse_terrain.elevations)
                    grad_ratio = min(drop / max_elev, 1.0) if max_elev > 0 else 0
                    windiness = 0.3 - grad_ratio * 0.25
                    iterations = int(5 - grad_ratio * 3)
                else:
                    windiness, iterations = 0.15, 3

                aPath = MapPath(points, style=StyleCSS("dummy"))
                windy = aPath.make_windy(iterations=iterations,
                                         offset_factor=windiness)
                inline = (f"fill:none;stroke:#1565c0;stroke-width:{avg_width:.1f};"
                          f"opacity:0.7;stroke-linecap:round;stroke-linejoin:round")
                parts.append(windy.drawSpline(adds=f'style="{inline}"'))

            # ── Lakes: project terminal basin ──
            if not shed.is_ocean and shed.terminal_hex is not None:
                lake = shed.lake_basin(base_size=lake_base_size, max_hexes=3)
                lake_fine = set()
                for lh in lake.hexes:
                    fi = _first_fine(c2f, lh)
                    if 0 <= fi < N:
                        lake_fine.add(fi)
                if lake_fine:
                    fine_region = HexRegion(hexes=lake_fine, hexGrid=grid)
                    lake_style = StyleCSS("lake_zoom", fill="#1565c0", stroke="none")
                    grid.builder.add_style(lake_style)
                    parts.append(fine_region.draw(
                        style=lake_style, inset=0.25,
                        f=unique_windy_edge(iterations=3)
                    ))

        return '\n'.join(parts)

    # No hard requires — the overlay checks internally and returns ""
    # if neither basins nor coarse_basins+c2f are available
    return OverlaySpec("rivers", render, requires=set(), priority=60)


So we have a fair amount of things build up to make a zoomed in DrainageBasin context. We want to make sure that we don't call the function that is n^2. for the river overlay we need a version that does the zoomed in context correctly.

So I am trying to do something like
```
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=6),
    terrain=myStuff.terr,
    basins=myStuff.basin,
    region=myStuff.region(),
    debug = not showDemo
)
```

maybe the bug is that it doesn't compute based upon the region

In [ ]:
!cat ../HexMagic/water/*.py

Do you have enough context to build a better TerrainDisplay and RiverOverlay?

TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

In [ ]:
??HexGrid.styleLayer

In [ ]:
myTerr.hexGrid.builder.layers = []
myTerr.hexGrid.builder.layers = []

In [ ]:
#| export
#| export
def CreamOverlay(stylized=False, edge_fn=None, **kw) -> OverlaySpec:
    """Parchment-style base fills. Pass stylized=True for organic region borders."""
    def render(ctx: OverlayContext) -> str:
        terrain, grid, builder = ctx.terrain, ctx.grid, ctx.builder

        terrain_fills = {
            'ocean':     '#E3F2FD',
            'coast':     '#F8F4E8',
            'lowland':   '#FDF5E6',
            'plains':    '#FAF0D4',
            'hills':     '#EFE6D5',
            'highlands': '#E8DFD0',
            'mountain':  '#DED4C4',
        }

        styles = {k: StyleCSS(k, fill=v) for k, v in terrain_fills.items()}
        for s in styles.values():
            builder.add_style(s)

        if 'distance_to_coast' not in terrain.fields:
            terrain.compute_distance_to_coast()

        for i in range(len(terrain.elevations)):
            elev = terrain.elevations[i]
            dist_coast = terrain.fields['distance_to_coast'][i]

            if elev <= 0:                              style = styles['ocean']
            elif dist_coast <= 2 and elev < 200:       style = styles['coast']
            elif elev < 200:                           style = styles['lowland']
            elif elev < 500:                           style = styles['plains']
            elif elev < 1200:                          style = styles['hills']
            elif elev < 2000:                          style = styles['highlands']
            else:                                      style = styles['mountain']

            grid.hexes[i].style = style

        return ""
    return OverlaySpec("cream", render, priority=10, stylized=stylized, edge_fn=edge_fn)


In [ ]:
#| export
#| export
def SoilOverlay(f=None, **kw) -> OverlaySpec:
    """Soil type hatch pattern overlay showing bedrock types."""
    def render(ctx: OverlayContext) -> str:
        soil = getattr(ctx, 'soil', None)
        if soil is None:
            soil = SoilSystem.from_plates(ctx.terrain, [])

        terrain, grid, builder = ctx.terrain, ctx.grid, ctx.builder
        patGen = TerrainPatterns(terrain)
        cols = [x.to_nc() for x in SoilType.standard_types()]

        # Ocean wave pattern
        ocean_hexes = terrain.find_region_at_level(0)
        ocean_region = HexRegion(hexes=ocean_hexes, hexGrid=grid)

        wave = patGen.wavePattern("ocean_waves_pat",
                                  amplitude=4, wavelength=16,
                                  color="#1565C0", fill="#E3F2FD")
        oceanStyle = StyleCSS("ocean", fill="url(#ocean_waves_pat)")
        builder.add_definition(wave)
        builder.add_style(oceanStyle)

        # Soil hatch patterns
        patterns, soilStyles = patGen.namedHatchPattern(cols, stroke_width=4, spacing=8)
        for p in patterns: builder.add_definition(p)
        for s in soilStyles: builder.add_style(s)

        # Set hex styles by soil region
        for i, region in enumerate(soil.regions):
            style = soilStyles[i]
            for h in region:
                grid.hexes[h].style = style

        for i in ocean_region:
            grid.hexes[i].style = oceanStyle

        return ""  # Pure hex style mutation — grid.update() bakes it in
    return OverlaySpec("soil", render, priority=15)


In [ ]:
#| export
#| export
def FlowOverlay(**kw) -> OverlaySpec:
    """Flow direction arrows showing drainage patterns."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.flow_diagram()
    return OverlaySpec("flow", render, priority=42)


In [ ]:
#| export
def TemperatureOverlay(ocean_color: str = "#2c5f8d", **kw) -> OverlaySpec:
    """Temperature gradient overlay — purple→blue→green→yellow→red."""
    def render(ctx: OverlayContext) -> str:
        return ctx.terrain.render_icon_temperature(ocean_color=ocean_color)
    return OverlaySpec("temperature", render, priority=41)


In [ ]:
#| export
#| export
def PrecipitationOverlay(levels: int = 5, min_density: float = 0.25, 
                          color: str = "#4A90E2", **kw) -> OverlaySpec:
    """Precipitation density dots — denser dots = more rainfall."""
    def render(ctx: OverlayContext) -> str:
        if 'precipitation' not in ctx.terrain.fields:
            ctx.terrain.compute_weather()
        return ctx.terrain.dotted_field(
            field='precipitation',
            flow_levels=levels,
            min_density=min_density,
            color=color
        )
    return OverlaySpec("precipitation", render, priority=43)


Do any of these others PrecipitationOverlay, TemperatureOverlay, FlowOverlay, SoilOverlay have zoomed in context issues?

In [ ]:
soil = SoilSystem.from_plates(myTerr, [])

In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    
    CreamOverlay(),
    SoilOverlay(),
    RiverOverlay(),
    soil=soil,       # ← pass it here

    
    basins=basins,
    terrain=myTerr,
    
    debug=True
)


I think this is all so strong. Can you do a design doc on how to use this?

In [ ]:
myTerr.hexGrid.builder.layers = []
TerrainDisplay(
    
    TerrainOverlay(),
    RiverOverlay(),
    basins=basins,
    terrain=myTerr,
    
    debug=False
)

In [ ]:
myTerr.hexGrid.builder.layers = []

In [ ]:
# Custom edge style
TerrainDisplay(TerrainOverlay(stylized=True),
               terrain=myTerr)

In [ ]:
show(myTerr.hexGrid.builder)

In [ ]:
# Flat hexes (default)

# Zoom into a region
peak = myTerr.find_peaks(1, 500)[0]
region = HexRegion(hexes=set(myTerr.ring(peak, 5)), hexGrid=myTerr.hexGrid)
TerrainDisplay(CreamOverlay(stylized=True), terrain=myTerr, region=region, padding=2)


In [ ]:
!cat ../HexMagic/plot/*.py

So I want two things. 
1. an option for TerrainOverlay and cream to uses stylized hex regions
2. a way for TerrainDisplay to zoom into a HexRegion

We aren't getting the stylized. we are just getting the hexes

In [ ]:
??HexTouchMap

Does the overlay guide at the top need to be updated at all. any errata or appendix?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call to audit it. Here's what I found comparing the doc against the actual implementation:

## Errata

**1. `OverlaySpec` has two new fields not documented:**

```python
@dataclass
class OverlaySpec:
    name: str
    renderer: Callable
    requires: set = field(default_factory=set)
    priority: int = 50
    stylized: bool = False       # ← missing from doc
    edge_fn: Callable = None     # ← missing from doc
```

**2. Phase 2 description is incomplete.** The doc says "grid.update() is called once" but the actual code branches:

```python
if stylized_spec:
    hexes_svg = grid.styleLayer(f=stylized_spec.edge_fn or unique_windy_edge(), smooth=True)
    grid.builder.adjust("hexes", hexes_svg)
else:
    grid.update()
```

So if *any* overlay has `stylized=True`, the hexes layer uses organic region boundaries instead of flat hex polygons.

**3. Overlay table is missing overlays and params:**

| Issue | Fix |
|-------|-----|
| `TerrainOverlay` and `CreamOverlay` — no mention of `stylized`, `edge_fn` params | Add them |
| `RiverOverlay` — missing `lake_base_size` param | Add it |
| `WatershedOverlay` (pri 35, requires `result`) — not listed | Add row |
| `SquadVisionOverlay` (pri 65) — not listed | Add row (or note it's game-specific) |

**4. Context table missing zoom keywords:**

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `region=` | triggers `terrain.zoom()` | Zoomed-in views |
| `padding=` | passed to `zoom()` (default 1) | Border hexes around region |
| `radius=` | calls `grid.adjustRadius()` | Custom hex sizing |

When `region=` is used with `basins=`, the basins get stashed as `ctx.coarse_basins` and `RiverOverlay` projects coarse river paths onto the fine grid via `c2f`.

**5. The Contract section should add:** Overlays with `stylized=True` — only the *first* one found (by priority) controls the edge function. Multiple stylized overlays don't compose; only one wins.

## Suggested Appendix

Two sections worth adding:

**A. Stylized Mode** — How `CreamOverlay(stylized=True)` or `TerrainOverlay(stylized=True)` replaces the flat hex grid with organic Gosper-curve boundaries. One-liner to show the difference.

**B. Zoom & Projection** — How `region=` works, what `c2f` is (coarse-to-fine index mapping), and how `RiverOverlay` handles the zoomed case by projecting coarse river segments onto fine grid coordinates. This is the trickiest part of the system and completely undocumented.

**C. Testing with `OverlayContext`** — The `from_terrain()` classmethod and `.test()` method for standalone overlay debugging outside of `TerrainDisplay`:

```python
ctx = OverlayContext.from_terrain(myTerr, basins=basins)
ctx.test(CreamOverlay(), RiverOverlay())  # prints ✓ per overlay, raises on failure
```

Want me to draft the updated doc with these fixes folded in?

Please write the upated doc with these fixes folded in

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```markdown
# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: If any overlay has `stylized=True`, the hexes layer is rendered with organic region boundaries via `grid.styleLayer()`. Otherwise `grid.update()` is called, flushing hex style mutations into the `hexes` layer as flat polygons.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

**Stylized note:** Only the *first* overlay (by priority) with `stylized=True` controls the edge function. Multiple stylized overlays don't compose — one wins.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay(stylized, edge_fn)` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay(stylized, edge_fn)` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `WatershedOverlay()` | 35 | SVG | Watershed boundary dots (requires `result`) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width, lake_base_size)` | 60 | SVG | River network from drainage basins |
| `SquadVisionOverlay(squads, num_turns, ...)` | 65 | SVG | Vision cones + facing bars (game-specific) |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Stylized mode — organic region borders

Pass `stylized=True` to any hex-style overlay to replace flat hex polygons with Gosper-curve organic boundaries:

```python
# Organic edges on terrain colors
TerrainDisplay(TerrainOverlay(stylized=True), terrain=t)

# Organic edges on parchment
TerrainDisplay(CreamOverlay(stylized=True), terrain=t)

# Custom edge function
TerrainDisplay(
    CreamOverlay(stylized=True, edge_fn=unique_windy_edge(iterations=4)),
    terrain=t
)
```

Without `stylized=True`, hexes render as clean polygons. With it, region boundaries get a hand-drawn feel via `grid.styleLayer()`.

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` (coarse mode) |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | `WatershedOverlay`, overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |
| `region=` | triggers `terrain.zoom(region, padding)` | Zoomed-in views |
| `padding=` | passed to `zoom()` (default `1`) | Border hexes around region |
| `radius=` | calls `grid.adjustRadius()` | Custom hex sizing |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'} — but actually requires=set(), checks internally
# If neither basins nor coarse_basins+c2f are available, returns ""
```

---

## Zoom & Projection

When you pass `region=` to `TerrainDisplay`, the system creates a zoomed-in view:

```python
TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(max_width=6),
    terrain=myTerr,
    basins=basins,
    region=my_region,   # ← triggers zoom
    padding=2
)
```

### What happens internally

1. `terrain.zoom(region, padding)` creates a new fine-resolution `Terrain` cropped to the region, with a `c2f` mapping (coarse index → list of fine indices).
2. The original `basins` reference the coarse grid, so they're stashed as `ctx.coarse_basins`. The `ctx.basins` field is set to `None` to prevent overlays from accidentally using stale coarse indices.
3. `c2f` is stored in `ctx.c2f` for overlays that need to project coarse data onto the fine grid.

### How RiverOverlay handles zoom

`RiverOverlay` detects the zoomed context (`c2f is not None and coarse_basins is not None`) and switches to projection mode:

- Iterates coarse river segments from `coarse_basins`
- Maps each coarse hex index to its fine grid position via `c2f`
- Draws the river path using fine grid coordinates with width scaling preserved from the coarse flow data
- Projects lake basins the same way

This avoids recomputing drainage on the fine grid (which would be O(n²)) and keeps rivers visually consistent between zoom levels.

---

## Creating Custom Overlays

### OverlaySpec fields

```python
@dataclass
class OverlaySpec:
    name: str                     # Unique layer name
    renderer: Callable            # fn(ctx: OverlayContext) -> str
    requires: set = set()         # Context keys needed (skip if missing)
    priority: int = 50            # Z-order (lower = bottom)
    stylized: bool = False        # Use organic region boundaries
    edge_fn: Callable = None      # Edge transform for stylized mode
```

### SVG overlay example

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Hex style mutation example

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## Testing with OverlayContext

Use `OverlayContext.from_terrain()` and `.test()` to debug overlays outside of `TerrainDisplay`:

```python
# Build a standalone context
ctx = OverlayContext.from_terrain(myTerr, basins=basins, soil=soil)

# Run overlays — errors propagate (not swallowed), prints status per overlay
ctx.test(CreamOverlay(), RiverOverlay())
#   ✓ cream                pri=10   style-only
#   ✓ rivers               pri=60   SVG (4823 chars)
```

`from_terrain()` accepts the same keywords as `TerrainDisplay` (`region=`, `padding=`, `radius=`, `basins=`, `soil=`, `result=`, `board=`, `c2f=`), so you can test zoom scenarios too:

```python
ctx = OverlayContext.from_terrain(myTerr, basins=basins, region=my_region, padding=2)
ctx.test(CreamOverlay(), RiverOverlay())
```

Unlike `TerrainDisplay` which logs warnings and skips, `.test()` raises `ValueError` on missing dependencies — fail fast for debugging.

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Stylized edges | `CreamOverlay(stylized=True)` or `TerrainOverlay(stylized=True)` |
| Custom edge function | `CreamOverlay(stylized=True, edge_fn=unique_windy_edge(iterations=4))` |
| Zoom into a region | `region=my_region, padding=2` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Test standalone | `OverlayContext.from_terrain(t).test(SomeOverlay())` |
| Create custom | Write a function returning `OverlaySpec` |
```